In [ ]:
import os
import pandas as pd
def load_authors_data(authors_file):
    columns = [
        "validation_status", "author.author_id", "affiliation",
        "email_domain", "total_citations", "author_id"
    ]
    authors_data = []
    with open(authors_file, "r", encoding="cp1252") as f:
        for line in f:
            split_data = line.strip().split('|')
            if len(split_data) == 6:
                authors_data.append(split_data)
            else:
                authors_data.append(split_data + [None] * (6 - len(split_data)))
    return pd.DataFrame(authors_data, columns=columns)

In [ ]:
import math
import numpy as np

def h_index(papers):
    citations = sorted([c for c, _ in papers], reverse=True)
    return sum(1 for i, c in enumerate(citations, 1) if c >= i)

def timed_h_index(papers, current_year, t=5):
    filtered = [(c, y) for c, y in papers if current_year - y < t]
    return h_index(filtered)

def contemporary_h_index(papers, current_year, gamma=4, delta=1):
    """
    Sidiropoulos et al. (2007)
    SC_i = gamma / (Y - Y_i + 1)^delta * C_i
    hC = max i s.t. SC_i >= i
    """
    if not papers:
        return 0
    sc_scores = []
    for c, y in papers:
        age = current_year - y + 1
        score = gamma / (age ** delta) * c if age > 0 else gamma * c
        sc_scores.append(score)

    sc_scores.sort(reverse=True)
    for i, sc in enumerate(sc_scores, 1):
        if sc < i:
            return i - 1
    return len(sc_scores)

def time_weighted_citation_index(papers, current_year, t=5, gamma=1.5):
    T1 = current_year - t + 1
    T2 = current_year
    score = 0.0
    for c, y in papers:
        if T1 <= y <= T2:
            age = T2 - y + 1
            score += np.log(c + 1) / (age ** gamma)
    return score

def calculate_all_indices(papers, current_year):
    return {
        "h": h_index(papers),
        "hc": contemporary_h_index(papers, current_year),
        "h_t5": timed_h_index(papers, current_year, t=5),
        "h_t10": timed_h_index(papers, current_year, t=10),
        "TWCI": time_weighted_citation_index(papers, current_year, t=5, gamma=1.0)
    }

    

In [ ]:
def load_author_publications(data_dir, author_id, start_year=2008, end_year=2013):
    """
    Load a single author's publication data as a DataFrame.
    Columns: [citations, year]
    Filters publications to a specific year range [start_year, end_year].
    """
    import os
    import pandas as pd

    file_path = os.path.join(data_dir, f"{author_id}_.dat")
    if not os.path.exists(file_path):
        return pd.DataFrame(columns=["citations", "year"])

    with open(file_path, encoding="cp1252") as f:
        lines = [line.strip().split("|") for line in f if "|" in line]
        # year 조건: start_year ≤ y ≤ end_year
        records = [(int(c), int(y)) for c, y in lines if start_year <= int(y) <= end_year]

    return pd.DataFrame(records, columns=["citations", "year"])


In [ ]:
def extract_summary_stats(df):
    if df.empty:
        return {
            "num_papers": 0,
            "total_citations": 0,
            "career_years": 0
        }

    return {
        "num_papers": len(df),
        "total_citations": df["citations"].sum(),
        "career_years": df["year"].max() - df["year"].min() + 1
    }


In [ ]:
def get_yearly_stats(df):
    if df.empty:
        return pd.DataFrame(columns=["year", "num_papers", "total_citations"])
    
    grouped = df.groupby("year").agg(
        num_papers=("citations", "count"),
        total_citations=("citations", "sum")
    ).reset_index()
    
    return grouped


In [ ]:
os.chdir("c:/Users/Hyeuk/Desktop/NetDB/Git/Expert_index_project/Expert_Index")
print(os.getcwd())

In [ ]:
# 설정
authors_file = "./gsc_data/authors.all"
data_dir = "./gsc_data/DATA/"
output_file = "time_weighted_index_ranking.csv"

start_year = 2008
end_year = 2013

In [ ]:
# 1. 지수 계산 및 로딩
print("📥 지수 계산 또는 로드 중...")
authors_data = load_authors_data(authors_file)
results = []
yearly_records = []

for _, row in authors_data.iterrows():
    author_id = row["author_id"]
    df = load_author_publications(data_dir, author_id, start_year, end_year)

    papers = list(map(tuple, df.values))
    if not papers:
        continue

    indices = calculate_all_indices(papers, end_year)
    summary = extract_summary_stats(df)
    yearly_df = get_yearly_stats(df)
    yearly_df["author_id"] = author_id
    
    yearly_records.append(yearly_df)

    result = {
        "author_id": author_id,
        **summary,
        **indices
    }
    results.append(result)
    


df_yearly = pd.concat(yearly_records, ignore_index=True)
df_yearly.to_csv("yearly_publication_stats.csv", index=False)
df = pd.DataFrame(results)

for col in ["h", "hc", "h_t5", "h_t10", "TWCI"]:
    df[f"rank_{col}"] = df[col].rank(ascending=False, method="min")
    
df.to_csv("time_weighted_index_ranking.csv", index=False)

In [ ]:

df_yearly = pd.concat(yearly_records, ignore_index=True)
df_yearly.to_csv("yearly_publication_stats.csv", index=False)
df = pd.DataFrame(results)

for col in ["h", "hc", "h_t5", "h_t10", "TWCI"]:
    df[f"rank_{col}"] = df[col].rank(ascending=False, method="min")
    
df.to_csv("time_weighted_index_ranking.csv", index=False)